# Workshop RNA-seq — Expressão diferencial em Mieloma Múltiplo

**Dado:** MMRF-CoMMpass (NCI Genomic Data Commons), quantificação STAR - Counts
**Contraste:** mieloma ao diagnóstico × mieloma na recidiva
**Ferramenta:** PyDESeq2

---

### Como usar este notebook

1. **Arquivo → Salvar uma cópia no Drive.** Trabalhe na sua cópia.
2. Execute as células **em ordem**, de cima para baixo (`Shift + Enter`).
3. Você não precisa escrever código. Quando for para mexer em alguma coisa, está marcado com 🔧.
4. Se algo quebrar: `Ambiente de execução → Reiniciar e executar tudo`.

> ⚠️ Este notebook é **material didático**. Ele demonstra o método; não é um estudo com rigor metodológico. As limitações estão listadas no final e devem ser lidas.

---
## M0 — Preparar o ambiente

Instala os pacotes na máquina virtual do Colab. Leva 1–2 minutos. Rode **uma vez** por sessão.

In [ ]:
%pip install -q pydeseq2 adjustText

import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pydeseq2

print("Python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("pydeseq2   :", pydeseq2.__version__)
print("\nAnote estas versões — elas entram na seção de métodos.")

---
## 🔧 Painel de controle

**Esta é a única célula que você vai mexer durante o workshop.** Cada vez que mudar
um valor aqui, rode desta célula para baixo.

In [ ]:
# ---- Amostragem ---------------------------------------------------------
N_POR_GRUPO = 30        # amostras por grupo. 30 roda em ~2 min. Não passe de 60 no Colab grátis.
SEED        = 42        # semente da subamostragem (reprodutibilidade)

# ---- Filtro de genes ----------------------------------------------------
MIN_CONTAGENS   = 10    # contagens mínimas...
MIN_AMOSTRAS    = None  # ...em pelo menos quantas amostras? None = usa o tamanho do menor grupo

REMOVER_IG = True       # remove segmentos V(D)J de imunoglobulina/TCR ANTES do teste.
                        # Cada clone de plasmócito tem seu próprio rearranjo:
                        # IGKV/IGHV/IGLV têm expressão idiossincrática por paciente
                        # e dominam o contraste por acaso. Em mieloma isto é praxe.
                        # ⚠️ Tem que ser IGUAL ao REMOVER_IG da trilha R, senão o
                        #    Módulo 9 compara universos de teste diferentes.

# ---- Amostras: travar pela lista do R (essencial para o Módulo 9) -------
USAR_AMOSTRAS_DO_R = True     # True  -> lê amostras_R.csv e usa as MESMAS amostras
                              # False -> sorteia por conta própria (as duas trilhas
                              #          vão analisar pacientes DIFERENTES)
CAMINHO_AMOSTRAS_R = "amostras_R.csv"

# ---- Significância ------------------------------------------------------
PADJ  = 0.05            # FDR
LFC   = 1.0             # |log2 fold change| mínimo  (1.0 = dobro)
TESTAR_LIMIAR_NO_MODELO = True   # True  -> testa H0: |LFC| <= LFC  (recomendado)
                                 # False -> testa H0: LFC == 0 e filtra depois (o jeito comum, pior)

# ---- Genes para destacar nos gráficos -----------------------------------
GENES_INTERESSE = ["MYC", "CCND1", "CCND2", "NSD2", "FGFR3", "MAF",
                   "TP53", "KRAS", "NRAS", "TNFRSF17", "SLAMF7", "CD38"]

# ---- Rótulos do contraste (confirme com a saída do M1) ------------------
MAPA_SAMPLE_TYPE = {
    "Primary Blood Derived Cancer - Bone Marrow":   "diagnostico",
    "Recurrent Blood Derived Cancer - Bone Marrow": "recidiva",
}
REFERENCIA = "diagnostico"   # grupo de referência do contraste
ALVO       = "recidiva"      # numerador do fold change

# ---- 🔧 Release do GDC: PREENCHA antes de rodar --------------------------
# Vai direto para o registro do M7. O GDC reprocessa os dados entre releases:
# sem esta linha, "baixado do GDC" não identifica nada.
# Veja em: gdc.cancer.gov/about-data/data-release
RELEASE_GDC = "Data Release 46.0 - August 10, 2026"

# ---- Plano B: matriz congelada pelo instrutor ---------------------------
# Publicada no Zenodo com DOI. O "?download=1" é obrigatório: sem ele o
# servidor devolve a página HTML de preview, não o arquivo.
DOI_DADOS    = "10.5281/zenodo.22399365"
URL_COUNTS   = "https://zenodo.org/records/22399365/files/mm_counts.csv.gz?download=1"
URL_METADATA = "https://zenodo.org/records/22399365/files/mm_metadata.csv?download=1"
URL_GENES    = "https://zenodo.org/records/22399365/files/gene_names.csv.gz?download=1"

np.random.seed(SEED)
print("Configuração carregada.")
if not RELEASE_GDC:
    print("⚠️ RELEASE_GDC está vazio — preencha acima antes de gerar o registro final (M7).")

---
## M1 — Onde mora o dado: consultando a API do GDC

O GDC guarda dados genômicos de câncer de vários projetos. Vamos perguntar a ele
quais arquivos de contagem existem no projeto **MMRF-COMMPASS**.

Repare que a consulta é um **filtro estruturado**: projeto + tipo de dado + workflow + acesso aberto.

### 🔍 Primeiro: que grupos existem neste dado?

Antes de baixar qualquer coisa, olhe os metadados. Esta é a diferença entre
fazer análise e apertar botões.

In [ ]:
import json, requests

GDC_FILES = "https://api.gdc.cancer.gov/files"
GDC_DATA  = "https://api.gdc.cancer.gov/data"

FILTROS = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.project_id",
                                 "value": ["MMRF-COMMPASS"]}},
        {"op": "in", "content": {"field": "data_type",
                                 "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "analysis.workflow_type",
                                 "value": ["STAR - Counts"]}},
        {"op": "in", "content": {"field": "access", "value": ["open"]}},
    ],
}

def facetas():
    """Pergunta ao GDC quantos arquivos existem por tipo de amostra."""
    params = {
        "filters": json.dumps(FILTROS),
        "facets": "cases.samples.sample_type",
        "size": "0",
        "format": "JSON",
    }
    r = requests.get(GDC_FILES, params=params, timeout=120)
    r.raise_for_status()
    buckets = r.json()["data"]["aggregations"]["cases.samples.sample_type"]["buckets"]
    return pd.DataFrame(buckets).rename(columns={"key": "sample_type", "doc_count": "n"})

try:
    tabela = facetas()
    print("Tipos de amostra disponíveis (arquivos de acesso aberto):\n")
    print(tabela.to_string(index=False))
    ONLINE = True
except Exception as e:
    print("Não consegui falar com o GDC agora:", type(e).__name__)
    print("Sem problema — vamos usar a matriz congelada pelo instrutor.")
    ONLINE = False

### 💬 Pare e discuta

Olhe a tabela acima.

- Existe algum grupo "normal" / "tecido saudável"? **Não.** Todas as amostras são de pacientes com mieloma.
- Isso muda a pergunta que podemos fazer. Não dá para perguntar *"o que muda no mieloma em relação ao plasmócito normal?"*. Dá para perguntar *"o que muda entre o diagnóstico e a recidiva?"*.
- Os nomes na coluna `sample_type` são os que você deve usar no `MAPA_SAMPLE_TYPE` do painel de controle. Se não baterem, **corrija lá em cima** e rode de novo.

> **Nota sobre desenho:** o CoMMpass é longitudinal — o mesmo paciente pode ter amostra
> no diagnóstico e na recidiva. Amostras do mesmo paciente **não são independentes**.
> A célula seguinte mantém **uma amostra por paciente** para respeitar a premissa do modelo.

In [ ]:
def listar_arquivos():
    params = {
        "filters": json.dumps(FILTROS),
        "fields": ",".join([
            "file_id", "file_name",
            "cases.case_id", "cases.submitter_id",
            "cases.samples.sample_type", "cases.samples.submitter_id",
        ]),
        "format": "JSON",
        "size": "20000",
    }
    r = requests.get(GDC_FILES, params=params, timeout=300)
    r.raise_for_status()
    linhas = []
    for h in r.json()["data"]["hits"]:
        caso    = h["cases"][0]
        amostra = caso["samples"][0]
        cond = MAPA_SAMPLE_TYPE.get(amostra["sample_type"])
        if cond is None:
            continue                      # descarta tipos que não entram no contraste
        linhas.append({
            "file_id":     h["file_id"],
            "case_id":     caso.get("submitter_id", caso["case_id"]),
            "barcode":     amostra.get("submitter_id", ""),
            "sample_type": amostra["sample_type"],
            "condition":   cond,
        })
    return pd.DataFrame(linhas)


def selecionar(tab):
    """Uma amostra por paciente + subamostragem balanceada."""
    tab = tab.sample(frac=1, random_state=SEED)                # embaralha
    tab = tab.drop_duplicates(subset="case_id", keep="first")  # 1 amostra por paciente
    partes = []
    for cond, bloco in tab.groupby("condition"):
        n = min(N_POR_GRUPO, len(bloco))
        partes.append(bloco.head(n))
    return pd.concat(partes).set_index("file_id")


def usar_amostras_do_R(tab, caminho):
    """Reusa EXATAMENTE as amostras da trilha R.

    ⚠️ POR QUE ISTO EXISTE — e é a lição mais importante do Módulo 9:
    `set.seed(42)` no R e `random_state=42` no pandas NÃO produzem o mesmo
    sorteio. São geradores pseudoaleatórios diferentes, com algoritmos de
    amostragem diferentes. O número 42 é o mesmo; a sequência que ele gera, não.

    Sem travar as amostras, as duas trilhas analisam pacientes DIFERENTES, e o
    Módulo 9 mede sorteio em vez de medir implementação. Já aconteceu: correlação
    de fold change de 0,37 e concordância de sinal de 61% (o acaso é 50%).
    """
    lista = pd.read_csv(caminho)
    assert "file_id" in lista.columns, f"{caminho} precisa ter a coluna 'file_id'"

    encontrados = tab[tab["file_id"].isin(lista["file_id"])].set_index("file_id")
    faltando = set(lista["file_id"]) - set(encontrados.index)
    if faltando:
        print(f"⚠️ {len(faltando)} amostra(s) do R não estão na consulta atual do GDC.")
        print("   O GDC reprocessa dados entre releases — os file_id mudam.")
    encontrados = encontrados.loc[[f for f in lista["file_id"] if f in encontrados.index]]
    print(f"✅ usando as MESMAS {len(encontrados)} amostras da trilha R")
    return encontrados


if ONLINE:
    todos = listar_arquivos()
    print(f"Arquivos elegíveis: {len(todos)}")
    print(todos["condition"].value_counts().to_string(), "\n")

    if USAR_AMOSTRAS_DO_R and os.path.exists(CAMINHO_AMOSTRAS_R):
        meta = usar_amostras_do_R(todos, CAMINHO_AMOSTRAS_R)
    else:
        if USAR_AMOSTRAS_DO_R:
            print(f"⚠️ {CAMINHO_AMOSTRAS_R} não encontrado — sorteando por conta própria.")
            print("   O Módulo 9 vai comparar amostras DIFERENTES. Suba o arquivo")
            print("   pelo painel 📁 e rode esta célula de novo.\n")
        meta = selecionar(todos)

    print("\nComposição final:")
    print(meta["condition"].value_counts().to_string())
    display(meta.head())


### Baixando os arquivos de contagem

Cada arquivo tem ~4 MB. Com 60 amostras, são ~2–4 minutos. Se você já rodou antes
nesta sessão, os arquivos já estão em disco e ele pula.

In [ ]:
import gzip
from io import StringIO
from pathlib import Path

RAW = Path("data_raw"); RAW.mkdir(exist_ok=True)

# ⚠️ A ARMADILHA QUE ESTA CÉLULA EVITA
# O GDC corta conexões longas. Um download interrompido deixa um arquivo PARCIAL
# no disco. Se a função só perguntar "o arquivo existe?", ela reaproveita o lixo
# na próxima execução: a matriz sai com NaN e o erro só aparece muito depois,
# dentro do PyDESeq2. Aqui o arquivo só é publicado se estiver íntegro.

LINHAS_ESPERADAS  = 60665   # GENCODE v36: comentário + cabeçalho + 4 N_* + 60.660 genes
TOLERANCIA_LINHAS = 10
TENTATIVAS        = 4


def arquivo_ok(caminho):
    p = Path(caminho)
    if not p.exists() or p.stat().st_size < 3e6:
        return False
    try:
        with open(p, "rb") as fh:
            n = sum(1 for _ in fh)
    except Exception:
        return False
    return abs(n - LINHAS_ESPERADAS) <= TOLERANCIA_LINHAS


def baixar(file_id):
    destino = RAW / f"{file_id}.tsv"
    if arquivo_ok(destino):
        return True                      # cache válido
    destino.unlink(missing_ok=True)      # cache inválido: descarta

    for k in range(1, TENTATIVAS + 1):
        tmp = destino.with_suffix(".parcial")
        try:
            with requests.get(f"{GDC_DATA}/{file_id}", stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(tmp, "wb") as fh:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        fh.write(chunk)
            if arquivo_ok(tmp):
                tmp.rename(destino)      # só publica se íntegro
                return True
        except Exception:
            pass
        tmp.unlink(missing_ok=True)
        if k < TENTATIVAS:
            time.sleep(2 ** k)
    return False


if ONLINE:
    import time
    falhas = []
    for i, fid in enumerate(meta.index, 1):
        if not baixar(fid):
            falhas.append(fid)
        if i % 10 == 0 or i == len(meta):
            print(f"  {i}/{len(meta)} arquivos")

    if falhas:
        print(f"\n⚠️ {len(falhas)} de {len(meta)} arquivos falharam após {TENTATIVAS} tentativas.")
        print("Descartando essas amostras e seguindo com o restante.")
        meta = meta.drop(index=falhas)
        print(meta["condition"].value_counts().to_string())
        if len(meta) == 0:
            ONLINE = False
            print("Nenhum arquivo baixado — indo para o plano B.")
        elif meta["condition"].value_counts().min() < 5:
            raise RuntimeError("Menos de 5 amostras em um dos grupos. Rode de novo "
                               "ou use a matriz congelada.")
        elif USAR_AMOSTRAS_DO_R:
            print("⚠️ As amostras não são mais idênticas às do R. O Módulo 9 vai")
            print("   comparar conjuntos levemente diferentes — anote isso.")
    else:
        print(f"Download concluído — {len(meta)} arquivos validados.")


---
## M2 — Da contagem à matriz

### Anatomia de um arquivo STAR-Counts

Antes de montar a matriz, **abra um arquivo** e olhe. Muita gente analisa RNA-seq
a vida inteira sem nunca ter olhado o arquivo bruto.

In [ ]:
if ONLINE:
    exemplo = list(RAW.glob("*.tsv"))[0]
    with open(exemplo) as fh:
        linhas = [next(fh) for _ in range(10)]
    print("".join(linhas))
    print("...")
    print("""
O que você está vendo:
  linha 1        -> comentário (# gene-model). pd.read_csv sozinho quebra aqui.
  linha 2        -> o cabeçalho de verdade: gene_id, gene_name, gene_type,
                    unstranded, stranded_first, stranded_second, tpm_..., fpkm_...
  linhas N_*     -> NÃO são genes. São o resumo do alinhamento do STAR:
                    N_unmapped, N_multimapping, N_noFeature, N_ambiguous.
                    Somá-las junto com os genes infla o tamanho da biblioteca.
  demais linhas  -> um gene por linha, Ensembl ID COM versão (ex.: ENSG00000141510.16)

Vamos usar a coluna 'unstranded' como contagem bruta, e descartar as linhas N_*.
""")

In [ ]:
USAR_COLUNA = "unstranded"

def _ler_star(caminho):
    """Lê o TSV do GDC pulando o comentário do topo; detecta gzip."""
    with open(caminho, "rb") as fh:
        gz = fh.read(2) == b"\x1f\x8b"
    abrir = gzip.open if gz else open
    with abrir(caminho, "rt", encoding="utf-8", errors="replace") as fh:
        linhas = fh.readlines()
    idx = next(i for i, l in enumerate(linhas) if l.startswith("gene_id"))
    return pd.read_csv(StringIO("".join(linhas[idx:])), sep="\t")


def montar_matriz(file_ids):
    series, mapa = {}, None
    for fid in file_ids:
        df = _ler_star(RAW / f"{fid}.tsv")
        df = df[~df["gene_id"].astype(str).str.startswith("N_")]
        if mapa is None:
            mapa = df[["gene_id", "gene_name", "gene_type"]].drop_duplicates().set_index("gene_id")
        s = df.set_index("gene_id")[USAR_COLUNA].astype("int64")
        s.name = fid
        series[fid] = s
    return pd.DataFrame(series).fillna(0).astype("int64"), mapa


if ONLINE:
    counts, mapa_genes = montar_matriz(meta.index.tolist())
else:
    # Plano B: matriz congelada
    if not URL_COUNTS:
        raise RuntimeError("Sem conexão com o GDC e sem URL de backup. Avise o instrutor.")

    def _ler_url(url):
        """O '?download=1' esconde a extensão .gz, então o pandas não infere a
        compressão sozinho — sem compression='gzip' o erro é um
        UnicodeDecodeError no byte 0x8b, que não diz nada a ninguém."""
        comp = "gzip" if ".gz" in url else None
        return pd.read_csv(url, index_col=0, compression=comp)

    counts = _ler_url(URL_COUNTS)
    meta   = _ler_url(URL_METADATA)

    # O mapa de símbolos vem junto. Sem ele o REMOVER_IG não roda — e aí o
    # Módulo 9 compararia universos de teste diferentes entre R e Python.
    mapa_genes = _ler_url(URL_GENES) if URL_GENES else None

    print(f"Usando a matriz congelada pelo instrutor (DOI {DOI_DADOS}).")
    if mapa_genes is None:
        print("⚠️ Sem URL_GENES: símbolos indisponíveis. Avise o instrutor.")

# Alinhamento — a checagem mais importante do notebook inteiro
counts = counts[[c for c in counts.columns if c in meta.index]]
meta   = meta.loc[counts.columns]
assert list(counts.columns) == list(meta.index), "Amostras desalinhadas!"

print(f"Matriz: {counts.shape[0]:,} genes x {counts.shape[1]} amostras")
print(meta["condition"].value_counts().to_string())
counts.iloc[:5, :4]

### 💬 Pare e discuta

A matriz tem cerca de **60 mil linhas**, mas o genoma humano tem ~20 mil genes codificantes.
Por quê?

Porque a anotação inclui pseudogenes, lncRNAs, miRNAs, genes processados… A coluna
`gene_type` diz qual é qual. A maior parte deles terá contagem zero ou quase — e vai
cair no filtro do próximo módulo.

---
## M3 — QC e exploração **antes** de qualquer teste

Esta é a regra mais importante do workshop: **você olha o dado antes de testar o dado.**
Um outlier ou um efeito de lote encontrado aqui vale mais do que qualquer refinamento
estatístico depois.

### 3.1 Tamanho de biblioteca

In [ ]:
bibl = counts.sum(axis=0) / 1e6
ordem = meta["condition"].sort_values().index

fig, ax = plt.subplots(figsize=(12, 4))
cores = meta.loc[ordem, "condition"].map({REFERENCIA: "#1b6ca8", ALVO: "#d1495b"})
ax.bar(range(len(ordem)), bibl.loc[ordem].values, color=cores.values)
ax.set_ylabel("Milhões de reads")
ax.set_xlabel("Amostras (agrupadas por condição)")
ax.set_title("Tamanho da biblioteca por amostra")
ax.set_xticks([])
plt.tight_layout(); plt.show()

razao = bibl.max() / bibl.min()
print(f"Menor: {bibl.min():.1f} M   Maior: {bibl.max():.1f} M   Razão máx/mín: {razao:.1f}x")

if razao <= 3:
    print("\n✅ Até ~3x é confortável — a normalização do DESeq2 dá conta.")
elif razao <= 6:
    print(f"\n⚠️ {razao:.1f}x está acima do confortável (~3x), mas ainda tratável.")
    print("   A normalização por mediana de razões é robusta a isso; o que NÃO é")
    print("   robusto é somar N_unmapped/N_multimapping junto (fizemos certo no M2).")
    print("   Fique de olho: se as bibliotecas maiores forem quase todas de um grupo,")
    print("   profundidade vira confundidor. Olhe as cores do gráfico acima.")
else:
    print(f"\n🚩 {razao:.1f}x é muito. Investigue antes de seguir: pode haver amostra")
    print("   com falha de sequenciamento ou de captura.")

### 3.2 Filtro de genes — 🔧 a primeira decisão real

Genes com quase nenhuma contagem só atrapalham: não têm poder para detectar nada e
ainda aumentam o peso da correção para testes múltiplos.

Compare dois critérios:

In [ ]:
n_menor_grupo = meta["condition"].value_counts().min()
min_amostras  = MIN_AMOSTRAS if MIN_AMOSTRAS is not None else n_menor_grupo

# --- trava: nenhum NaN pode chegar ao modelo ------------------------------
n_nan = int(counts.isna().sum().sum())
if n_nan:
    ruins = counts.columns[counts.isna().any()].tolist()
    raise ValueError(
        f"{n_nan:,} valores NaN na matriz, em {len(ruins)} amostra(s).\n"
        f"Arquivos truncados no download: {', '.join(a[:8] for a in ruins)}\n"
        f"Apague-os de data_raw/ e rode a célula de download de novo."
    )

# Critério A — o do pipeline original: soma total
keep_A = counts.sum(axis=1) >= MIN_CONTAGENS

# Critério B — recomendado (estilo filterByExpr/edgeR): presença consistente
keep_B = (counts >= MIN_CONTAGENS).sum(axis=1) >= min_amostras

print(f"Genes totais                                  : {counts.shape[0]:,}")
print(f"A) soma total >= {MIN_CONTAGENS:<3}                        : {keep_A.sum():,}")
print(f"B) >= {MIN_CONTAGENS} contagens em >= {min_amostras} amostras       : {keep_B.sum():,}")
print(f"\nDiferença: {keep_A.sum() - keep_B.sum():,} genes que o critério A deixa passar")
print("(um gene com 10 reads numa única amostra passa em A, não passa em B)")

counts_f = counts.loc[keep_B]
print(f"\nCritério B: {counts_f.shape[0]:,} genes.")

# --- Segmentos V(D)J de imunoglobulina ------------------------------------
# Cada paciente de mieloma tem um clone de plasmócitos com rearranjo V(D)J
# ÚNICO. IGKV/IGHV/IGLV ficam com expressão altíssima e idiossincrática por
# amostra, e dominam o topo da lista por acaso — não por biologia de recidiva.
#
# ⚠️ Isto acontece AQUI, antes do DeseqDataSet, e não na hora de plotar.
#    Filtrar só no gráfico não muda o teste: o FDR já foi corrigido com esses
#    genes dentro, e o padj de todos os outros sai errado.
#
# 💬 Rode uma vez com REMOVER_IG = False e olhe o volcano. Depois volte para
#    True. A diferença entre os dois gráficos é a aula inteira.

IG_PADRAO = r"^(IGKV|IGHV|IGLV|IGKJ|IGHJ|IGLJ|IGKC|IGHG|IGHA|IGHM|IGHD|IGLC|TRBV|TRAV|TRGV|TRDV)"

if REMOVER_IG and mapa_genes is not None:
    simbolos = mapa_genes.reindex(counts_f.index)["gene_name"].astype("string")
    eh_ig = simbolos.str.match(IG_PADRAO).fillna(False)
    print(f"Segmentos V(D)J removidos: {int(eh_ig.sum()):,} genes")
    counts_f = counts_f.loc[~eh_ig.values]
    print(f"Matriz final para o teste: {counts_f.shape[0]:,} genes.")
elif REMOVER_IG:
    print("⚠️ REMOVER_IG=True mas não há mapa_genes (matriz congelada sem símbolos).")
    print("   Os segmentos de Ig vão para o teste — espere IGKV/IGHV no topo.")


### 3.3 Normalização, transformação e PCA

Contagem bruta tem um problema: a variância depende da média. Gene muito expresso
varia muito em valor absoluto, e domina qualquer PCA ou heatmap feito sobre o dado cru.

A solução é a **VST** (*variance stabilizing transformation*). Não é `log2(x+1)` —
é uma transformação calculada a partir da relação média-variância do próprio dado.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds  import DeseqStats

# PyDESeq2 espera amostras nas LINHAS e genes nas COLUNAS
X = counts_f.T
meta_dds = meta[["condition"]].copy()
meta_dds["condition"] = pd.Categorical(meta_dds["condition"],
                                       categories=[REFERENCIA, ALVO])

def criar_dds(X, meta_dds):
    """Compatível com as várias versões da API do PyDESeq2."""
    try:
        return DeseqDataSet(counts=X, metadata=meta_dds, design="~condition", refit_cooks=True)
    except TypeError:
        return DeseqDataSet(counts=X, metadata=meta_dds, design_factors="condition", refit_cooks=True)

dds = criar_dds(X, meta_dds)
dds.deseq2()
print("Modelo ajustado.")

# --- transformação para os gráficos ---
try:
    dds.vst()
    vst = pd.DataFrame(dds.layers["vst_counts"], index=dds.obs_names, columns=dds.var_names).T
    transf = "VST"
except Exception:
    normed = pd.DataFrame(dds.layers["normed_counts"], index=dds.obs_names, columns=dds.var_names).T
    vst = np.log2(normed + 1)
    transf = "log2(normalizado + 1)"
print("Transformação usada nos gráficos:", transf)

In [ ]:
from sklearn.decomposition import PCA

# PCA sobre os 2000 genes mais variáveis (padrão da área)
top_var = vst.var(axis=1).sort_values(ascending=False).head(2000).index
M = vst.loc[top_var].T                      # amostras x genes

p = PCA(n_components=2).fit(M)
pcs = p.transform(M)
var = p.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(7, 6))
for cond, cor in [(REFERENCIA, "#1b6ca8"), (ALVO, "#d1495b")]:
    m = (meta.loc[M.index, "condition"] == cond).values
    ax.scatter(pcs[m, 0], pcs[m, 1], s=70, alpha=.8, c=cor, label=cond, edgecolors="white")
ax.set_xlabel(f"PC1 ({var[0]:.1f}% da variância)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}% da variância)")
ax.set_title(f"PCA — {transf}, top 2000 genes variáveis")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### 💬 Pare e discuta — e não se assuste

**Os grupos separam?** Provavelmente **não**, ou não muito. E está tudo bem.

Isso não é falha do pipeline. A diferença entre diagnóstico e recidiva é sutil
comparada à heterogeneidade **entre pacientes**: mieloma tem subtipos citogenéticos
muito distintos — t(4;14), t(11;14), hiperdiploidia — e a maior fonte de variação
do PC1 é quase sempre essa, não o momento da doença.

Duas lições importantes:

1. **PCA que não separa ≠ ausência de genes diferenciais.** Significa que o efeito é
   menor que a variação entre indivíduos. O modelo estatístico ainda pode encontrá-lo,
   porque ele testa gene a gene em vez de olhar a variância global.
2. Se o PC1 separasse por **sexo** ou por **data de processamento**, isso seria um
   confundidor, e o modelo precisaria incluí-lo como covariável (`~ sex + condition`).

---
## M4 — Expressão diferencial

### O que o DESeq2 faz, em quatro etapas

1. **Normalização por mediana de razões.** Não é dividir pelo total de reads. Para cada gene,
   calcula-se a razão em relação à média geométrica entre amostras; o fator de tamanho de cada
   amostra é a **mediana** dessas razões.
   *Isso importa muito em mieloma:* plasmócitos produzem imunoglobulina em quantidade
   absurda (`IGHG1`, `IGKC`, `IGLC1`). Uma normalização por soma total deixaria esses
   poucos genes sequestrarem a escala de todas as amostras. A mediana é imune a isso.

2. **Estimativa de dispersão com *shrinkage*.** Genes com expressão parecida "emprestam"
   informação uns dos outros. É o que torna o método utilizável com poucas réplicas.

3. **Teste de Wald** por gene sobre o log2 fold change.

4. **Correção BH + filtragem independente.** Alguns genes voltam com `padj = NA` — não é erro,
   é decisão do algoritmo (baixa expressão, ou outlier pela distância de Cook).

In [ ]:
# --- Etapa de teste ------------------------------------------------------
kwargs = {"contrast": ["condition", ALVO, REFERENCIA]}

if TESTAR_LIMIAR_NO_MODELO:
    # Testa H0: |LFC| <= LFC. Controla o FDR PARA A PERGUNTA QUE VOCÊ ESTÁ FAZENDO.
    try:
        st = DeseqStats(dds, lfc_null=LFC, alt_hypothesis="greaterAbs", **kwargs)
        modo = f"H0: |log2FC| <= {LFC} (limiar dentro do teste)"
        LIMIAR_NO_TESTE = True
    except TypeError:
        st = DeseqStats(dds, **kwargs)
        modo = "H0: log2FC = 0 (versão do pydeseq2 não aceita lfc_null)"
        LIMIAR_NO_TESTE = False
else:
    st = DeseqStats(dds, **kwargs)
    modo = "H0: log2FC = 0 (limiar aplicado depois — o jeito comum)"
    LIMIAR_NO_TESTE = False

st.summary()
res = st.results_df.copy()
print("\nModo de teste:", modo)

# --- O QUE CONTA COMO SIGNIFICATIVO --------------------------------------
# Uma definição só, usada em TODOS os gráficos e tabelas daqui para frente.
#
#   limiar DENTRO do teste  -> o padj já responde "|LFC| > limiar?".
#                              Filtrar por |LFC| de novo é redundante e ainda
#                              usaria o LFC ENCOLHIDO, que não foi o testado.
#   limiar DEPOIS do teste  -> o padj responde "LFC ≠ 0?". O filtro por tamanho
#                              de efeito passa a ser necessário, mas o padj
#                              deixa de controlar o FDR da afirmação feita.
#                              É o jeito comum e é o pior.
def eh_significativo(df, coluna_lfc="log2FoldChange"):
    ok = df["padj"].notna() & (df["padj"] < PADJ)
    if not LIMIAR_NO_TESTE:
        ok &= df[coluna_lfc].abs() >= LFC
    return ok

CRITERIO = (f"padj < {PADJ} (o limiar de {LFC} já está no teste)" if LIMIAR_NO_TESTE
            else f"padj < {PADJ} E |log2FC| >= {LFC} (filtro post-hoc)")
print("Critério de DEG:", CRITERIO)


# --- COMO O padj APARECE NOS GRÁFICOS E TABELAS ----------------------------
# Notação científica ("1.00e+00") é ilegível e não é assim que se reporta um
# p-valor. A convenção de artigo:
#   >= 0,001 -> o valor com três casas          padj = 0,039
#   <  0,001 -> o limite                        padj < 0,001
# Reportar "3,3e-29" como se fosse uma medida precisa é fingir uma precisão
# que o método não tem.
def fmt_padj(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "< 0,001"
    return f"{p:.3f}".replace(".", ",")


def estrelas(p):
    """Convenção usual: *** < 0,001  ** < 0,01  * < 0,05  ns caso contrário."""
    if pd.isna(p):   return "ns"
    if p < 0.001:    return "***"
    if p < 0.01:     return "**"
    if p < 0.05:     return "*"
    return "ns"


def rotulo_padj(p):
    """Texto pronto para título de gráfico: 'padj = 0,039 *'."""
    sinal = "" if (not pd.isna(p) and p < 0.001) else "= "
    return f"padj {sinal}{fmt_padj(p)}  {estrelas(p)}"

print("Estrelas: *** padj < 0,001   ** < 0,01   * < 0,05   ns = não significativo")


In [ ]:
# ============================================================================
# M4b — Shrinkage do fold change
# ============================================================================
# Regra: teste com o LFC bruto, mas ranqueie e plote com o LFC encolhido.
#
# ⚠️ Consequência disso: a tabela final mostra o LFC ENCOLHIDO ao lado de um
#    p-valor calculado sobre o LFC BRUTO. Na maioria dos genes a diferença é
#    cosmética. Em genes de dispersão alta, não é — e a linha passa a parecer
#    contraditória. Por isso guardamos as DUAS colunas: sem elas, o aluno vê
#    "log2FC = -0.01, padj = 1e-13" e não tem como entender de onde veio.

def achar_coef(dds, alvo, ref):
    """O nome do coeficiente muda entre versões do pydeseq2."""
    candidatos = [f"condition[T.{alvo}]", f"condition_{alvo}_vs_{ref}"]
    try:
        cols = list(dds.obsm["design_matrix"].columns)
    except Exception:
        cols = []
    for c in candidatos:
        if c in cols:
            return c
    for c in cols:
        if alvo in c:
            return c
    return candidatos[0]


# Nota: esta célula depende de `res`, `st` e `dds`, criados no M4. Se der
# NameError, é porque o M4 não rodou nesta sessão — rode desde o M0.
coef = achar_coef(dds, ALVO, REFERENCIA)
print("Coeficiente:", coef)

res_bruto = res.copy()
try:
    st.lfc_shrink(coeff=coef)
    res = st.results_df.copy()

    # --- guarda o bruto ao lado do encolhido ------------------------------
    res["log2FC_bruto"] = res_bruto["log2FoldChange"]
    res["lfcSE_bruto"]  = res_bruto["lfcSE"]
    print("Shrinkage aplicado. Colunas log2FC_bruto e lfcSE_bruto adicionadas.")

    # --- sanidade: o shrinkage encolheu DEMAIS? ---------------------------
    razao_se = res_bruto["lfcSE"].median() / res["lfcSE"].median()
    print(f"lfcSE mediano: {res_bruto['lfcSE'].median():.4f} (bruto) → "
          f"{res['lfcSE'].median():.4f} (encolhido) = {razao_se:.1f}x")
    if razao_se > 10:
        print("\n🚩 O lfcSE colapsou. Sinal clássico de shrinkage aplicado duas vezes.")
        print("   Reinicie a sessão e rode desde o M0.")

    # --- efeito nos genes de MENOR expressão ------------------------------
    comp = pd.DataFrame({
        "LFC_bruto":     res_bruto["log2FoldChange"],
        "LFC_encolhido": res["log2FoldChange"],
        "baseMean":      res["baseMean"],
    }).dropna().sort_values("baseMean").head(10)
    print("\nEfeito do shrinkage nos genes de MENOR expressão:")
    print(comp.round(2).to_string())

    # --- onde o shrinkage foi MAIS agressivo ------------------------------
    # Aqui moram os genes cuja linha na tabela final parece contraditória.
    enc = (res_bruto["log2FoldChange"].abs() - res["log2FoldChange"].abs())
    top_enc = enc.dropna().sort_values(ascending=False).head(5).index
    detalhe = pd.DataFrame({
        "LFC_bruto":     res_bruto.loc[top_enc, "log2FoldChange"],
        "SE_bruto":      res_bruto.loc[top_enc, "lfcSE"],
        "LFC_encolhido": res.loc[top_enc, "log2FoldChange"],
        "pvalue":        res.loc[top_enc, "pvalue"],
        "baseMean":      res.loc[top_enc, "baseMean"],
    })
    if mapa_genes is not None:
        detalhe.insert(0, "gene_name", mapa_genes.reindex(top_enc)["gene_name"])
    print("\nOnde o shrinkage foi MAIS agressivo:")
    print(detalhe.round(3).to_string())
    print("""
💬 Estes são os genes cuja linha na tabela final parece se contradizer: LFC quase
   zero ao lado de um p-valor minúsculo. Não é bug. O p vem do LFC bruto; o LFC
   exibido é o encolhido. Quando o apeglm encolhe MUITO, é porque a estimativa
   bruta tinha pouca informação por trás — dispersão alta, ou uma única amostra
   extrema carregando o efeito.

   Rode plotar_gene() em um destes (M5.4). O boxplot mostra na hora.""")

except RuntimeError:
    raise
except Exception as e:
    print("Shrinkage indisponível nesta versão:", type(e).__name__)
    res["log2FC_bruto"] = res["log2FoldChange"]
    res["lfcSE_bruto"]  = res["lfcSE"]


### O *shrinkage* do fold change

Um gene com 12 reads pode ter `log2FoldChange = +8`. Isso quase sempre é ruído.
O `lfc_shrink` encolhe o efeito na proporção da incerteza da estimativa.

**Regra:** teste com o LFC bruto, mas **ranqueie e plote com o LFC encolhido.**

In [ ]:
# --- anotar com símbolo do gene ------------------------------------------
if mapa_genes is not None:
    res = res.join(mapa_genes, how="left")
elif "gene_name" not in res.columns:
    res["gene_name"] = res.index

res = res.sort_values("padj")

sig  = res[eh_significativo(res)]
up   = sig[sig["log2FoldChange"] > 0]
down = sig[sig["log2FoldChange"] < 0]

print(f"Genes testados                      : {res['padj'].notna().sum():,}")
print(f"Genes com padj = NA (filtrados)     : {res['padj'].isna().sum():,}")
print(f"Critério                            : {CRITERIO}")
print(f"DEGs                                : {len(sig):,}")
print(f"   aumentados em '{ALVO}'   : {len(up):,}")
print(f"   reduzidos  em '{ALVO}'   : {len(down):,}")
_prop = 100 * len(sig) / max(res["padj"].notna().sum(), 1)
print(f"\nProporção do transcriptoma testado: {_prop:.3f}%")

print("\n--- Top 15 por padj ---")
cols = ["gene_name", "baseMean", "log2FoldChange", "padj"]
if "log2FC_bruto" in res.columns:
    cols.insert(3, "log2FC_bruto")     # bruto ao lado do encolhido
print(res[cols].head(15).round(3).to_string())

### 💬 Pare e discuta — o teste de sanidade mais útil que existe

Olhe a **proporção do transcriptoma** que saiu significativa.

- **Menos de ~5%:** plausível para um contraste sutil como este.
- **Mais de ~25%:** bandeira vermelha. Quando um quarto do transcriptoma é "diferencial",
  o contraste quase sempre está capturando diferença de **tecido** ou de **composição celular**,
  não regulação gênica fina. (Para comparação: no pipeline original de colangiocarcinoma,
  tumor × tecido biliar normal deu 10.774 DEGs em 38.846 genes — 28%. É um resultado real,
  mas o que ele mede é sobretudo "fígado tumoral não é ducto biliar normal".)

### 🔧 Experimente agora

Volte ao **painel de controle** e:
1. Mude `PADJ` de `0.05` para `0.01`. Rode de novo o M4. Quantos DEGs sobraram?
2. Mude `TESTAR_LIMIAR_NO_MODELO` para `False`. Rode de novo. O número mudou muito?

O item 2 é a diferença entre *testar* a hipótese "o efeito é maior que 2×" e
*filtrar* pelo resultado depois. Não são a mesma coisa, e só o primeiro controla o
FDR para a pergunta que você quer responder.

---
## M5 — Visualização honesta

### 5.1 MA plot — o gráfico mais informativo e o menos usado

In [ ]:
df = res.dropna(subset=["padj", "log2FoldChange"]).copy()
df["sig"] = eh_significativo(df)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(df.loc[~df.sig, "baseMean"], df.loc[~df.sig, "log2FoldChange"],
           s=6, c="#cccccc", alpha=.5)
ax.scatter(df.loc[df.sig, "baseMean"], df.loc[df.sig, "log2FoldChange"],
           s=8, c="#d1495b", alpha=.8)
ax.set_xscale("log")
ax.axhline(0, c="black", lw=.8)
ax.axhline(LFC, ls="--", c="grey", lw=.8); ax.axhline(-LFC, ls="--", c="grey", lw=.8)
ax.set_xlabel("Expressão média normalizada (baseMean, escala log)")
ax.set_ylabel(f"log2 FC ({ALVO} / {REFERENCIA})")
ax.set_title("MA plot")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("Como ler: a nuvem deve estar centrada em zero (normalização OK).")
print("À esquerda (baixa expressão) a dispersão é maior — é aí que o shrinkage age.")

### 5.2 Volcano

In [ ]:
# ============================================================================
# 5.2 Volcano + painel dos genes de interesse
# ============================================================================
import logging
from adjustText import adjust_text
# O adjustText avisa sobre FancyArrowPatch em toda figura. É cosmético
# e assusta a turma sem motivo.
logging.getLogger("adjustText").setLevel(logging.ERROR)

# --- segmentos V(D)J: expressão idiossincrática por paciente ---------------
# Cada clone de plasmócito tem seu rearranjo. Dominam o contraste por acaso.
# Os segmentos V(D)J já saíram no filtro do M3 (REMOVER_IG), antes do teste.

d = res.dropna(subset=["padj", "log2FoldChange"]).copy()
d["gene_name"] = d["gene_name"].astype(str)

d["y"]   = -np.log10(d["padj"].clip(lower=1e-300))
d["sig"] = eh_significativo(d)
up, down = d.sig & (d.log2FoldChange > 0), d.sig & (d.log2FoldChange < 0)
print(f"DEGs após limpeza: {int(up.sum())} ↑ / {int(down.sum())} ↓")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(14, 6.5),
                             gridspec_kw={"width_ratios": [1.35, 1]})

# ---------------- painel A: volcano ---------------------------------------
lim  = float(np.ceil(max(d.loc[d.sig, "log2FoldChange"].abs().max(), LFC + 1))) + .5
ymax = float(d["y"].max()) * 1.10
dentro = d["log2FoldChange"].abs() <= lim

ax.axhline(-np.log10(PADJ), ls=":", c="0.7", lw=1)
for v in (-LFC, LFC):
    ax.axvline(v, ls=":", c="0.7", lw=1)
ax.scatter(d.loc[~d.sig, "log2FoldChange"], d.loc[~d.sig, "y"],
           s=13, c="#e2e2e2", lw=0, zorder=2)
ax.scatter(d.loc[down, "log2FoldChange"], d.loc[down, "y"], s=30, c="#2c6e9b",
           lw=0, zorder=3, label=f"↓ em {ALVO} ({int(down.sum())})")
ax.scatter(d.loc[up, "log2FoldChange"], d.loc[up, "y"], s=30, c="#c1435a",
           lw=0, zorder=3, label=f"↑ em {ALVO} ({int(up.sum())})")

# rotula APENAS significativos — os de interesse vão para o painel B
rot = d[d.sig & dentro].sort_values("padj").head(10).index
textos = [ax.text(d.at[i, "log2FoldChange"], d.at[i, "y"], d.at[i, "gene_name"],
                  fontsize=9, zorder=5) for i in rot]
if textos:
    adjust_text(textos, ax=ax, expand_points=(1.7, 1.9),
                arrowprops=dict(arrowstyle="-", color="#999", lw=.7))

ax.set_xlim(-lim, lim); ax.set_ylim(-ymax * .04, ymax)
ax.set_xlabel(f"log2 fold change  ({ALVO} / {REFERENCIA})", fontsize=10.5)
ax.set_ylabel("−log10 (padj)", fontsize=10.5)
ax.set_title("A · transcriptoma", fontsize=11.5, loc="left", pad=10)
ax.legend(frameon=False, fontsize=9, loc="upper left")

# ---------------- painel B: genes de interesse, com incerteza -------------
g = res[res["gene_name"].isin(GENES_INTERESSE)].copy()
g = g.dropna(subset=["log2FoldChange"]).sort_values("log2FoldChange")
g["ic"] = 1.96 * g["lfcSE"]                    # intervalo de confiança de 95%
y = np.arange(len(g))

bx.axvspan(-LFC, LFC, color="#f0f0f0", zorder=0)     # zona de efeito irrelevante
bx.axvline(0, c="0.35", lw=1, zorder=1)
cores = np.where(g["padj"] < PADJ, "#c1435a", "#4a4a4a")
bx.errorbar(g["log2FoldChange"], y, xerr=g["ic"], fmt="none",
            ecolor="#b0b0b0", elinewidth=1.6, capsize=3, zorder=2)
bx.scatter(g["log2FoldChange"], y, s=55, c=cores, zorder=3)

bx.set_yticks(y); bx.set_yticklabels(g["gene_name"], fontsize=9.5)
bx.set_ylim(-.7, len(g) - .3)
bx.set_xlim(-LFC * 2.2, LFC * 2.2)
bx.set_xlabel("log2 fold change (IC 95%)", fontsize=10.5)
bx.set_title("B · genes canônicos de mieloma", fontsize=11.5, loc="left", pad=10)

# baseMean à direita: mostra que são todos altamente expressos
for yi, bm in zip(y, g["baseMean"]):
    bx.text(1.02, yi, f"{bm:,.0f}", transform=bx.get_yaxis_transform(),
            va="center", fontsize=8, color="0.5")
bx.text(1.02, len(g) - .1, "baseMean", transform=bx.get_yaxis_transform(),
        va="center", fontsize=8, color="0.5", fontweight="bold")

for a in (ax, bx):
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color("0.75")
    a.tick_params(colors="0.45", labelsize=9)
    a.set_axisbelow(True)
ax.grid(axis="y", color="0.94", lw=.8)
bx.grid(axis="x", color="0.94", lw=.8)

fig.suptitle(f"Mieloma múltiplo — {ALVO} vs {REFERENCIA}", fontsize=14,
             x=.007, ha="left", y=.99)
fig.text(.007, .945, f"MMRF-CoMMpass · n={len(meta)} · sem segmentos de Ig · "
                     f"padj<{PADJ}, |LFC|≥{LFC}", fontsize=8.5, color="0.5")
plt.tight_layout(rect=[0, 0, 1, .93])
plt.savefig("volcano.png", dpi=200, bbox_inches="tight")
plt.show()

print("\nPainel B — todas as barras cruzam o zero e ficam dentro da faixa cinza.")
print("Os drivers do mieloma já estão expressos no diagnóstico e permanecem")
print("expressos na recidiva. Ausência de mudança, não ausência de expressão —")
print("é por isso que a coluna baseMean está ali.")

### 5.3 Heatmap — e o alerta mais importante da aula

⚠️ **Um heatmap dos "top N genes por padj" com clustering das amostras separa os grupos
por construção.** Você selecionou os genes *porque* eles separam os grupos, e depois
mostrou que eles separam os grupos. Isso é raciocínio circular, e é o gráfico mais
comum e mais enganoso em artigo de expressão.

O heatmap abaixo está aqui como demonstração. Leia o comentário no final.

In [ ]:
top = res.dropna(subset=["padj"]).head(30)
mat = vst.loc[vst.index.intersection(top.index)]
rot = top["gene_name"].reindex(mat.index).fillna(pd.Series(mat.index, index=mat.index))
mat.index = rot.values

z = mat.sub(mat.mean(axis=1), axis=0).div(mat.std(axis=1) + 1e-9, axis=0)
ordem = meta["condition"].sort_values().index
z = z[ordem]
cores_col = meta.loc[ordem, "condition"].map({REFERENCIA: "#1b6ca8", ALVO: "#d1495b"})

g = sns.clustermap(z, cmap="RdBu_r", center=0, figsize=(11, 9),
                   col_cluster=False, col_colors=cores_col.values,
                   yticklabels=True, xticklabels=False,
                   cbar_kws={"label": "z-score"})
g.fig.suptitle(f"Top 30 genes por padj — {transf}, z-score por gene", y=1.01)
plt.show()

print("""
⚠️  Este heatmap NÃO é evidência de nada. Os genes foram escolhidos exatamente
    por separarem os grupos. Se você quiser um heatmap que signifique algo:
      - selecione genes por VARIÂNCIA (independente do contraste), ou
      - use um conjunto de genes definido A PRIORI (uma assinatura publicada).
""")

### 5.4 O gráfico que desmente — contagem de um gene só

Aqui a mentira morre. Se o efeito "significativo" vem de duas amostras extremas,
o boxplot mostra na hora.

In [ ]:
# ============================================================================
# 5.4 O gráfico que desmente — contagem gene a gene
# ============================================================================
# Uma lista de DEGs é uma afirmação estatística. O boxplot das contagens é o
# que mostra se ela se sustenta: se o efeito vem de duas ou três amostras
# extremas, aparece na hora.
#
# Esta célula plota TODOS os genes significativos da sua rodada, não um
# exemplo escolhido a dedo. É a diferença entre demonstrar e conferir.

def plotar_gene(nome, ax=None, salvar=False):
    """Boxplot das contagens de um gene, com os pontos individuais por cima."""
    linhas = res.index[res["gene_name"] == nome]
    if len(linhas) == 0 or linhas[0] not in vst.index:
        print(f"'{nome}' não está entre os genes testados (pode ter caído no filtro).")
        return None
    gid = linhas[0]
    r = res.loc[gid]
    d = pd.DataFrame({"expr": vst.loc[gid], "grupo": meta["condition"]})

    sozinho = ax is None
    if sozinho:
        fig, ax = plt.subplots(figsize=(5, 5))

    sns.boxplot(data=d, x="grupo", y="expr", ax=ax, width=.5,
                palette={REFERENCIA: "#1b6ca8", ALVO: "#d1495b"}, fliersize=0,
                order=[REFERENCIA, ALVO], hue="grupo", legend=False)
    sns.stripplot(data=d, x="grupo", y="expr", ax=ax, color="black",
                  size=4, alpha=.6, jitter=.2, order=[REFERENCIA, ALVO])

    ax.set_title(f"{nome}\nlog2FC = {r['log2FoldChange']:+.2f}   ·   "
                 f"{rotulo_padj(r['padj'])}", fontsize=10.5)
    ax.set_ylabel(f"Expressão ({transf})" if sozinho else "")
    ax.set_xlabel("")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=9)

    if sozinho:
        plt.tight_layout()
        if salvar:
            plt.savefig(f"gene_{nome}.png", dpi=200, bbox_inches="tight")
        plt.show()
    return ax


# --- TODOS os significativos, em painéis ----------------------------------
alvos = list(sig["gene_name"].dropna().astype(str).unique())

if len(alvos) == 0:
    print("Nenhum gene significativo nesta configuração — nada para conferir aqui.")
    print("Isso também é um resultado. Rode plotar_gene() num gene à sua escolha:")
    plotar_gene(res["gene_name"].iloc[0])

else:
    MAX_PAINEIS = 12          # 🔧 acima disso a figura fica alta demais
    if len(alvos) > MAX_PAINEIS:
        print(f"{len(alvos)} genes significativos — mostrando os {MAX_PAINEIS} de "
              f"menor padj.\nPara ver os outros: plotar_gene(\"NOME\").\n")
        alvos = list(sig.sort_values("padj")["gene_name"]
                        .dropna().astype(str).unique())[:MAX_PAINEIS]
    else:
        print(f"{len(alvos)} gene(s) significativo(s) — olhando um por um:\n")
    print(res[res["gene_name"].isin(alvos)]
          [["gene_name", "baseMean", "log2FC_bruto", "log2FoldChange"]]
          .assign(padj=lambda d: d.index.map(lambda g: fmt_padj(res.at[g, "padj"])))
          .round(3).to_string(index=False))

    n = len(alvos)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 4.2 * rows),
                             squeeze=False)
    for k, nome in enumerate(alvos):
        plotar_gene(nome, ax=axes[k // cols][k % cols])
    for k in range(n, rows * cols):                 # apaga os painéis vazios
        axes[k // cols][k % cols].axis("off")
    axes[0][0].set_ylabel(f"Expressão ({transf})")

    fig.suptitle(f"Genes significativos — {CRITERIO}", fontsize=12,
                 fontweight="bold", y=1.005 if rows == 1 else 1.002)
    plt.tight_layout()
    plt.savefig("genes_significativos.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("""
💬 Olhe cada painel antes de reportar. O que você quer ver: duas nuvens de
   pontos deslocadas uma em relação à outra. O que deve acender o alerta: a
   maioria dos pontos sobreposta e dois ou três lá em cima puxando a média.

   Um gene que passou no teste mas tem esse padrão não deveria entrar no seu
   resumo — ou entra com a ressalva escrita.""")


# 🔧 Para olhar qualquer outro gene, troque o nome:
#    plotar_gene("MYC")
#    plotar_gene("TNFRSF17", salvar=True)


In [ ]:
# ============================================================================
# 5.5 — Figura composta
# ============================================================================
from matplotlib.gridspec import GridSpec
from scipy.cluster.hierarchy import linkage, dendrogram

# Os segmentos V(D)J já saíram no filtro do M3 (REMOVER_IG), antes do teste.

d = res.dropna(subset=["padj", "log2FoldChange"]).copy()
d["gene_name"] = d["gene_name"].astype(str)
d["y"]   = -np.log10(d["padj"].clip(lower=1e-300))
d["sig"] = eh_significativo(d)
up, down = d.sig & (d.log2FoldChange > 0), d.sig & (d.log2FoldChange < 0)

VERM, VERD, CINZA = "#c1435a", "#2a9d8f", "#dcdcdc"

fig = plt.figure(figsize=(16, 13))
gs  = GridSpec(3, 2, figure=fig, height_ratios=[1, .08, .95],
               hspace=.35, wspace=.22, top=.90, bottom=.10)

# ---------------- A · volcano com histogramas marginais -------------------
gsA = gs[0, 0].subgridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                           hspace=.05, wspace=.05)
ax   = fig.add_subplot(gsA[1, 0])
axhx = fig.add_subplot(gsA[0, 0], sharex=ax)
axhy = fig.add_subplot(gsA[1, 1], sharey=ax)

lim = float(np.ceil(d["log2FoldChange"].abs().quantile(.9995))) + .5
ax.axhline(-np.log10(PADJ), ls="--", c="0.7", lw=.9)
for v in (-LFC, LFC):
    ax.axvline(v, ls="--", c="0.7", lw=.9)
ax.scatter(d.loc[~d.sig, "log2FoldChange"], d.loc[~d.sig, "y"], s=12, c=CINZA, lw=0)
ax.scatter(d.loc[down, "log2FoldChange"], d.loc[down, "y"], s=28, c=VERD, lw=0,
           label=f"↓ em {ALVO}: {int(down.sum())}")
ax.scatter(d.loc[up, "log2FoldChange"], d.loc[up, "y"], s=28, c=VERM, lw=0,
           label=f"↑ em {ALVO}: {int(up.sum())}")

for i in d[d.sig].sort_values("padj").head(8).index:
    ax.annotate(d.at[i, "gene_name"], (d.at[i, "log2FoldChange"], d.at[i, "y"]),
                xytext=(5, 4), textcoords="offset points", fontsize=8.5)

axhx.hist(d["log2FoldChange"], bins=90, color="0.55", lw=0)
axhy.hist(d["y"], bins=90, orientation="horizontal", color="0.55", lw=0)
for a in (axhx, axhy):
    a.axis("off")
ax.set_xlim(-lim, lim)
ax.set_xlabel(f"log2 fold change ({ALVO} / {REFERENCIA})", fontsize=10)
ax.set_ylabel("−log10 (padj)", fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper left")
axhx.set_title(f"A. Volcano: {int(d.sig.sum())} genes diferencialmente expressos",
               fontsize=12, fontweight="bold", loc="left", pad=8)

# ---------------- B · heatmap SEM clusterizar amostras --------------------
bx = fig.add_subplot(gs[0, 1])
top = d[d.sig].sort_values("padj").head(30) if d.sig.sum() >= 5 \
      else d.sort_values("padj").head(30)
ids = [g for g in top.index if g in vst.index]
Z = vst.loc[ids]
Z = Z.sub(Z.mean(axis=1), axis=0).div(Z.std(axis=1).replace(0, 1), axis=0)

ordem = meta.sort_values("condition").index                # ordem fixa, sem clustering
Z = Z[ordem]
Z = Z.iloc[dendrogram(linkage(Z, "average"), no_plot=True)["leaves"]]  # só genes

im = bx.imshow(Z.values, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
bx.set_yticks(range(len(Z)))
bx.set_yticklabels(top["gene_name"].reindex(Z.index).fillna(pd.Series(Z.index, index=Z.index)),
                   fontsize=7.5)
bx.set_xticks([])
n_ref = int((meta.loc[ordem, "condition"] == REFERENCIA).sum())
bx.axvline(n_ref - .5, c="k", lw=1.6)
bx.text(n_ref / 2, -1.4, REFERENCIA, ha="center", fontsize=10,
        color=VERD, fontweight="bold")
bx.text((n_ref + len(ordem)) / 2, -1.4, ALVO, ha="center", fontsize=10,
        color=VERM, fontweight="bold")
bx.set_xlabel("amostras (ordenadas por condição, sem clusterização)", fontsize=9.5)
bx.set_title("B. Heatmap honesto: os grupos NÃO se separam",
             fontsize=12, fontweight="bold", loc="left", pad=22)
fig.colorbar(im, ax=bx, fraction=.02, pad=.12).set_label("z-score", fontsize=8.5)

# ---------------- C · alvos canônicos -------------------------------------
cx = fig.add_subplot(gs[2, :])
g = res[res["gene_name"].isin(GENES_INTERESSE)].dropna(subset=["log2FoldChange"])
g = g.sort_values("log2FoldChange")
ids_c = [i for i in g.index if i in vst.index]
g = g.loc[ids_c]
H = vst.loc[ids_c, ordem]
H = H.sub(H.mean(axis=1), axis=0).div(H.std(axis=1).replace(0, 1), axis=0)

im2 = cx.imshow(H.values, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
cx.set_yticks(range(len(g)))
cx.set_yticklabels([f"{n}  (LFC {l:+.2f}, {'sig' if p < PADJ else 'ns'})"
                    for n, l, p in zip(g["gene_name"], g["log2FoldChange"], g["padj"])],
                   fontsize=9.5)
cx.set_xticks([]); cx.axvline(n_ref - .5, c="k", lw=1.6)
cx.set_xlabel(f"amostras  ({REFERENCIA}: {n_ref}  |  {ALVO}: {len(ordem) - n_ref})",
              fontsize=10)
cx.set_title("C. Alvos canônicos de mieloma: expressos nos dois momentos, sem alteração",
             fontsize=12, fontweight="bold", loc="left", pad=10)
fig.colorbar(im2, ax=cx, fraction=.012, pad=.02).set_label("z-score", fontsize=8.5)

fig.suptitle(f"Mieloma múltiplo: {ALVO} vs {REFERENCIA}", fontsize=17,
             fontweight="bold", y=.965)
fig.text(.5, .932, f"MMRF-CoMMpass (GDC) · n={len(meta)} · segmentos V(D)J removidos · "
                   f"padj<{PADJ}, |LFC|≥{LFC}", ha="center", fontsize=10.5, color="0.4")
fig.text(.5, .045,
         "Figura. Expressão diferencial (PyDESeq2) entre mieloma ao diagnóstico e na recidiva. "
         "(A) Volcano com histogramas marginais.\n(B) Top 30 genes por padj, amostras em ordem fixa "
         "— clusterizar amostras selecionadas por significância separa os grupos por construção.\n"
         "(C) Alvos canônicos, com log2FC e significância no rótulo. Dados públicos do NCI GDC.",
         ha="center", fontsize=9, color="0.3")

plt.savefig("figura_composta.png", dpi=200, bbox_inches="tight")
plt.show()

### 🔧 Experimente

1. Rode `plotar_gene()` com o gene **do topo** da sua lista.
2. Rode com um gene **não significativo** (padj alto). Veja a diferença.
3. Rode com `"TNFRSF17"` (BCMA — alvo de CAR-T e biespecíficos em mieloma) e com `"CD38"`
   (alvo do daratumumabe). Eles mudam entre diagnóstico e recidiva?

Essa última pergunta tem relevância clínica direta: perda de antígeno é um mecanismo
conhecido de escape terapêutico.

---
## M6 — Da lista ao significado biológico

### 6.1 Genes de interesse em mieloma

In [ ]:
alvo_tab = res[res["gene_name"].isin(GENES_INTERESSE)][
    ["gene_name", "baseMean", "log2FoldChange", "padj"]
].copy()

# Usa a MESMA definição do resto do notebook (M4). Um "não significativo"
# aqui e um "significativo" no volcano seria o pior tipo de inconsistência:
# silenciosa e plausível.
_sig_alvo = eh_significativo(alvo_tab)

def classificar(gid):
    if pd.isna(alvo_tab.at[gid, "padj"]):
        return "filtrado / NA"
    return "significativo" if _sig_alvo[gid] else "não significativo"

if not alvo_tab.empty:
    alvo_tab["status"] = [classificar(g) for g in alvo_tab.index]
    print(alvo_tab.sort_values("padj").round(3).to_string(index=False))
else:
    print("Nenhum dos genes de interesse sobreviveu ao filtro.")

print("""
💬 Discussão: se MYC ou os genes de translocação (NSD2, FGFR3, MAF, CCND1) não
   aparecerem como diferenciais, isso NÃO significa que eles não importam no mieloma.
   Significa que a relevância deles é ESTRUTURAL — translocação, amplificação,
   posicionamento junto ao enhancer de IGH — e não necessariamente transcricional
   entre estes dois momentos da doença.

   É exatamente a mesma lição do FGFR2 em colangiocarcinoma: uma análise só de
   expressão não captura alteração estrutural. Daí o valor da abordagem multiômica.
""")

### 6.2 Exportar para análise de enriquecimento

Duas coisas saem daqui:

- `deseq_results.csv` — a tabela completa, para o seu registro
- `degs_para_enriquecimento.txt` — só os símbolos, para colar no ShinyGO
- `genes_de_fundo.txt` — **o universo correto**: os genes que entraram no teste,
  não "todos os genes humanos"

O universo de fundo errado é a causa nº 1 de enriquecimento inflado.

In [ ]:
from google.colab import files as colab_files

res.to_csv("deseq_results.csv")

lista_degs = (sig["gene_name"].dropna().astype(str).unique())
fundo      = (res.loc[res["padj"].notna(), "gene_name"].dropna().astype(str).unique())

pd.Series(lista_degs).to_csv("degs_para_enriquecimento.txt", index=False, header=False)
pd.Series(fundo).to_csv("genes_de_fundo.txt", index=False, header=False)

# matriz com símbolo, para a Trilha B (iDEP)
if mapa_genes is not None:
    m = counts_f.join(mapa_genes[["gene_name"]])
    m = m[m["gene_name"].notna() & (m["gene_name"] != "")]
    m = m.groupby("gene_name").sum(numeric_only=True)
    m.to_csv("mm_counts_symbols.csv")
    meta[["condition"]].to_csv("mm_metadata_idep.csv")

print(f"DEGs exportados : {len(lista_degs):,}")
print(f"Genes de fundo  : {len(fundo):,}")
print("\nArquivos gerados — use o painel de Arquivos (📁) à esquerda para baixar,")
print("ou descomente a linha abaixo.")
# colab_files.download("degs_para_enriquecimento.txt")

### 6.3 🔧 Agora vá para a Trilha B

**No ShinyGO** — `https://bioinformatics.sdstate.edu/go/`
1. Cole o conteúdo de `degs_para_enriquecimento.txt`
2. Em *Custom background*, cole `genes_de_fundo.txt` ← **não pule este passo**
3. Espécie: *Human*. Rode.

**No iDEP 2.0** — `https://bioinformatics.sdstate.edu/idep20/`
1. Suba `mm_counts_symbols.csv` e `mm_metadata_idep.csv`
2. *Pre-Process*: mesmo filtro, transformação VST
3. *DEG1*: método DESeq2, FDR 0,05, fold change mínimo 2
4. **Compare o número de DEGs e o top-20 com o que saiu aqui no Colab**

💬 **A pergunta que fecha o workshop:** os dois caminhos deram o mesmo resultado?
Se deram, o achado é robusto. Se não deram, *por quê*? (Filtro diferente,
transformação diferente, versão diferente do DESeq2.) Descobrir isso é o que
separa quem sabe rodar de quem sabe confiar.

⚠️ **Cuidado com termos genéricos.** Quase toda lista grande de DEGs em câncer
enriquece "ciclo celular". Termos com mais de 500 genes enriquecem por inércia.
Olhe o *fold enrichment* e o tamanho do conjunto, não só o p-valor.

---
## M7 — Reprodutibilidade: a seção de métodos

Rode a célula abaixo e **guarde a saída**. Sem isso, sua análise não é reproduzível —
o GDC reprocessa os dados periodicamente, e "baixado do GDC" não identifica nada.

In [ ]:
# ============================================================================
# M7 — Registro da análise (a seção de métodos que você vai colar no artigo)
# ============================================================================
# Este bloco DESCREVE o que foi feito nesta execução — não o que o notebook
# poderia ter feito. Cada linha é lida das variáveis reais, nunca escrita à mão.
#
# 💬 Um registro que diz "subamostragem aleatória, semente 42" quando a execução
#    leu uma lista fixa é pior que registro nenhum: é uma descrição errada com
#    aparência de precisão. O módulo sobre reprodutibilidade não pode ser o que
#    mente sobre o próprio pipeline.

from datetime import datetime

# RELEASE_GDC vem do painel de controle (M0).

# --- descrições derivadas do que REALMENTE aconteceu -----------------------
_travadas = USAR_AMOSTRAS_DO_R and os.path.exists(CAMINHO_AMOSTRAS_R)
_selecao = (f"1 amostra/paciente; lista fixa lida de '{CAMINHO_AMOSTRAS_R}' "
            f"(mesmas amostras da trilha R)"
            if _travadas else
            f"1 amostra/paciente; subamostragem aleatória (numpy/pandas, semente {SEED})")

_n_ig = int(eh_ig.sum()) if (REMOVER_IG and mapa_genes is not None) else 0
_ig = (f"sim — {_n_ig:,} segmentos V(D)J removidos ANTES do teste"
       if _n_ig else
       "não — segmentos V(D)J incluídos no teste")

# mesma correção da guarda do M4b: dir() devolve o escopo LOCAL e falha aqui.
_shrink = "sim (lfc_shrink / apeglm-MAP)" if "log2FC_bruto" in res.columns else "não"
_release = RELEASE_GDC if RELEASE_GDC else "<<< NÃO PREENCHIDO — veja gdc.cancer.gov/about-data/data-release >>>"
_prop = 100 * len(sig) / max(res["padj"].notna().sum(), 1)

print(f"""
=== REGISTRO DA ANÁLISE (trilha Python) ===
Data de execução      : {datetime.now():%d/%m/%Y %H:%M}
Fonte                 : NCI GDC, projeto MMRF-COMMPASS
Workflow              : STAR - Counts, coluna '{USAR_COLUNA}'
Release do GDC        : {_release}

Desenho
  Contraste           : {ALVO} vs {REFERENCIA} (referência = {REFERENCIA})
  n por grupo         : {meta['condition'].value_counts().to_dict()}
  Seleção             : {_selecao}

Processamento
  Genes antes filtro  : {counts.shape[0]:,}
  Filtro de expressão : >= {MIN_CONTAGENS} contagens em >= {min_amostras} amostras
  Remoção de V(D)J    : {_ig}
  Genes testados      : {counts_f.shape[0]:,}
  Transformação (EDA) : {transf}

Estatística
  Modelo              : ~condition
  Teste               : {modo}
  Shrinkage do LFC    : {_shrink}
  Correção múltipla   : Benjamini-Hochberg
  Critério de DEG     : {CRITERIO}
  DEGs                : {len(sig):,} ({_prop:.3f}% dos genes testados)

Versões
  Python              : {sys.version.split()[0]}
  pydeseq2            : {pydeseq2.__version__}
  pandas              : {pd.__version__}
  numpy               : {np.__version__}
""")

if not RELEASE_GDC:
    print("⚠️ Preencha RELEASE_GDC no topo desta célula. O GDC reprocessa os dados")
    print("   entre releases: sem essa linha, 'baixado do GDC' não identifica nada.")

if _travadas:
    print("✅ Amostras travadas pela lista do R — as duas trilhas são comparáveis.")
else:
    print("⚠️ Amostras sorteadas nesta trilha. set.seed(42) no R e random_state=42")
    print("   no pandas NÃO dão o mesmo conjunto — o Módulo 9 vai medir sorteio.")

### Limitações deste desenho — o que **precisa** estar escrito

1. **Não há plasmócito normal.** Nada aqui permite afirmar o que difere entre mieloma
   e tecido normal. O contraste é entre dois momentos da mesma doença.
2. **O tratamento é um confundidor não controlado.** Entre o diagnóstico e a recidiva
   o paciente recebeu terapia. Parte do que aparece como "biologia da progressão"
   é resposta ao tratamento, e as duas coisas não são separáveis neste desenho.
3. **Pureza variável da amostra.** A fração de plasmócitos tumorais na seleção CD138+
   difere entre pacientes. Diferença de expressão pode ser diferença de composição celular.
4. **Sem correção de lote nem de covariáveis clínicas** (idade, sexo, subtipo citogenético,
   estádio ISS, centro de coleta).
5. **Sem coorte de validação independente.**
6. **Subamostragem didática:** usamos apenas parte da coorte, escolhida aleatoriamente.

### ✍️ Exercício

Escreva, em **cinco linhas**, a seção de métodos desta análise. Passe para o colega ao
lado. Ele tenta apontar o que falta para reproduzir o resultado sem falar com você.

---
## M8 — Desafio final: o controle positivo

Existe um teste que valida seu pipeline inteiro de ponta a ponta: **contraste por sexo.**

Se você comparar amostras masculinas e femininas, os genes de topo *têm* que ser
`XIST` (inativação do X, alto em mulheres) e os genes ligados ao Y — `RPS4Y1`, `DDX3Y`,
`UTY`, `KDM5D`, `EIF1AY`.

Se o seu pipeline **não** encontra isso, ele está errado em algum lugar — e é melhor
descobrir agora do que no artigo.

> Para rodar, você precisa dos metadados clínicos do GDC (endpoint `/cases`,
> campo `demographic.gender`). O código abaixo busca e monta o contraste.

In [ ]:
# ============================================================================
# M8 — Controle positivo: contraste por sexo (rótulo por votação)
# ============================================================================
# O GDC não libera demographic.gender para MMRF-CoMMpass no tier aberto — os dois
# endpoints (/cases e /files) devolvem None. Alternativa: inferir o sexo pela
# expressão dos genes do Y e validar o pipeline em XIST.
#
# Por que votação e não um corte no escore médio: cortar pelo "maior intervalo"
# sempre devolve algum corte, mesmo sem dois grupos de verdade. Aqui cada gene
# classifica as amostras por conta própria e comparamos os votos. Genes
# independentes do mesmo cromossomo TÊM que concordar — a concordância é a
# evidência, não o formato do histograma.
#
# ⚠️ Controle DEGRADADO: o rótulo saiu da matriz de expressão, então todos os
# genes do Y ficaram circulares e estão fora do veredito. Quem valida é XIST.

from sklearn.cluster import KMeans

GENES_Y = ["RPS4Y1", "DDX3Y", "UTY", "KDM5D", "EIF1AY", "USP9Y", "TXLNGY", "ZFY", "NLGN4Y"]
CONCORDANCIA_MIN = 0.80     # fração mínima de genes que devem concordar, por amostra
PROSSEGUIR_MESMO_ASSIM = False   # 🔧 só ligue se o diagnóstico abaixo convencer você


def ids_do_gene(nome):
    """gene_ids do GDC trazem sufixo de versão (ENSG00000129824.16)."""
    if mapa_genes is not None:
        alvo = mapa_genes.index[mapa_genes["gene_name"] == nome]
    else:
        alvo = [g for g in vst.index if str(g).split(".")[0] == nome or g == nome]
    return [g for g in alvo if g in vst.index]


usados = {g: ids_do_gene(g)[0] for g in GENES_Y if ids_do_gene(g)}
print(f"Genes do Y disponíveis ({len(usados)}/{len(GENES_Y)}): {list(usados)}")
assert len(usados) >= 3, (
    "Menos de 3 genes do Y sobreviveram ao filtro do M3. Baixe MIN_CONTAGEM/"
    "MIN_AMOSTRAS no painel de controle e rode de novo a partir do M3."
)

# --- cada gene classifica as amostras sozinho ------------------------------
votos, forca = {}, {}
for nome, gid in usados.items():
    v = vst.loc[gid].astype(float)
    km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(v.values.reshape(-1, 1))
    centros = km.cluster_centers_.ravel()
    alto = int(np.argmax(centros))                       # cluster de maior expressão = male
    votos[nome] = pd.Series((km.labels_ == alto).astype(int), index=v.index)
    forca[nome] = abs(centros[1] - centros[0])           # distância entre os centros

votos = pd.DataFrame(votos).T                            # genes x amostras
consenso = votos.mean(axis=0)                            # fração de genes que votou "male"
rotulo = pd.Series(np.where(consenso > 0.5, "male", "female"),
                   index=consenso.index, name="sexo")

# concordância por amostra: 1.0 = todos os genes disseram a mesma coisa
concordancia = np.maximum(consenso, 1 - consenso)

print("\n--- diagnóstico por gene ---")
diag = pd.DataFrame({
    "separacao_clusters": pd.Series(forca).round(2),
    "n_male": votos.sum(axis=1).astype(int),
    "concorda_c_consenso": (votos.eq((rotulo == "male").astype(int), axis=1)
                            .mean(axis=1).round(3)),
}).sort_values("concorda_c_consenso", ascending=False)
print(diag.to_string())

print(f"\nrótulo final: {rotulo.value_counts().to_dict()}")
print(f"concordância média : {concordancia.mean():.3f}")
print(f"concordância mínima: {concordancia.min():.3f}")
ambiguas = concordancia[concordancia < CONCORDANCIA_MIN]
print(f"amostras ambíguas (< {CONCORDANCIA_MIN:.0%}): {len(ambiguas)} de {len(rotulo)}")

# --- inspeção visual -------------------------------------------------------
escore = vst.loc[list(usados.values())].astype(float).mean(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(12, 3))

for grupo, cor in [("female", "#d1495b"), ("male", "#1b6ca8")]:
    v = escore[rotulo == grupo]
    ax[0].scatter(v, np.random.normal(0, .06, len(v)), c=cor, s=45, alpha=.75, label=grupo)
ax[0].set_yticks([]); ax[0].legend(frameon=False)
ax[0].set_xlabel(f"escore Y — média VST de {len(usados)} genes")
ax[0].set_title("Escore agregado", fontsize=10)

ax[1].hist(concordancia, bins=np.linspace(0.5, 1.0, 11), color="#3d5a80", edgecolor="w")
ax[1].axvline(CONCORDANCIA_MIN, ls="--", c="#d1495b", lw=1.5)
ax[1].set_xlabel("concordância entre genes, por amostra")
ax[1].set_ylabel("n amostras")
ax[1].set_title("Votos alinhados = rótulo confiável", fontsize=10)
plt.tight_layout(); plt.show()

# --- checagens -------------------------------------------------------------
ok = (concordancia.mean() >= CONCORDANCIA_MIN) and (rotulo.nunique() == 2)
if not ok and not PROSSEGUIR_MESMO_ASSIM:
    raise AssertionError(
        f"Os genes do Y não concordam entre si (média {concordancia.mean():.2f}). "
        f"Olhe a tabela de diagnóstico: se um gene específico destoa, remova-o de "
        f"GENES_Y. Se todos destoam, o sinal de Y não está presente nesta "
        f"amostragem — veja a nota ao final da célula."
    )

assert rotulo.nunique() == 2, f"Um grupo só: {rotulo.value_counts().to_dict()}"
n_min = rotulo.value_counts().min()
assert n_min >= 3, f"Só {n_min} amostra(s) no menor grupo — aumente N_POR_GRUPO."

# --- modelo ----------------------------------------------------------------
ms = rotulo.to_frame().astype(str)
Xs = counts_f[ms.index].T                 # amostras nas LINHAS, genes nas COLUNAS
assert Xs.shape[0] == len(ms), f"Xs={Xs.shape[0]} amostras, ms={len(ms)} — desalinhado."
assert Xs.shape[0] > 0 and Xs.shape[1] > 0, f"Matriz vazia: {Xs.shape}"
print(f"\nmatriz do contraste: {Xs.shape[0]} amostras x {Xs.shape[1]:,} genes\n")

dds_s = DeseqDataSet(counts=Xs, metadata=ms, design="~sexo", refit_cooks=True)
dds_s.deseq2()
ss = DeseqStats(dds_s, contrast=["sexo", "male", "female"])
ss.summary()

rs = ss.results_df.copy()
if mapa_genes is not None:
    rs = rs.join(mapa_genes[["gene_name"]], how="left")
elif "gene_name" not in rs.columns:
    rs["gene_name"] = rs.index

print("\n=== TOP 15 — contraste por sexo ===")
print(rs.sort_values("padj")[["gene_name", "log2FoldChange", "padj"]]
        .head(15).round(3).to_string())

# --- veredito --------------------------------------------------------------
# Os genes do Y definiram os grupos: achá-los no topo é tautologia. Quem valida
# é XIST, que é do X e não participou da rotulagem.
xist = rs[rs["gene_name"] == "XIST"]

print("\n" + "=" * 62)
if len(xist):
    lfc, padj = float(xist["log2FoldChange"].iloc[0]), float(xist["padj"].iloc[0])
    print(f"XIST — o teste que vale: log2FC = {lfc:+.2f} | {rotulo_padj(padj)}")
    if lfc < -1 and padj < 0.05:
        print("\n✅ Pipeline validado. XIST reprimido no grupo 'male', como tem que")
        print("   estar. É do cromossomo X, não entrou na rotulagem: evidência")
        print("   independente de que o pipeline funciona de ponta a ponta.")
    else:
        print("\n❌ XIST não se comportou como esperado. Revise:")
        print("   (a) o alinhamento counts × meta no M2")
        print("   (b) a direção do contraste (male vs female)")
        print("   (c) a tabela de diagnóstico acima — grupos podem estar trocados")
else:
    print("⚠️ XIST não sobreviveu ao filtro do M3 — sem evidência independente.")
    print("   Os genes do Y no topo são circulares e NÃO validam nada.")
print("=" * 62)

---
# M9 — Concordância R × Python

Este é o módulo que justifica ter feito a análise duas vezes.

Você rodou o **DESeq2** (R, Bioconductor, a implementação de referência) e o
**PyDESeq2** (Python, um porte independente do mesmo método). Mesmo dado, mesmo
contraste, mesmos limiares. A única diferença foi a linguagem.

> **A pergunta:** duas implementações independentes do mesmo método estatístico
> chegam ao mesmo resultado?

Se chegam, o achado é da biologia, não da ferramenta. Se divergem em algum ponto,
descobrir *onde* e *por quê* ensina mais do que qualquer slide.

### 🔧 Suba os DOIS arquivos do R

No painel de Arquivos (📁) à esquerda, faça upload de:

- `resultados/deseq_results_R.csv` — a tabela de resultados
- `resultados/amostras_R.csv` — a lista de amostras

O segundo é o que garante que as duas trilhas analisaram **os mesmos pacientes**.
Sem ele, `set.seed(42)` no R e `random_state=42` no pandas sorteiam conjuntos
diferentes, e a comparação mede o sorteio em vez da implementação.

In [ ]:
# Opção 1 — upload manual pelo painel de Arquivos (📁)
# Opção 2 — descomente para abrir o seletor de arquivos:
# from google.colab import files as _f; _f.upload()

CAMINHO_R = "deseq_results_R.csv"

if not os.path.exists(CAMINHO_R):
    raise FileNotFoundError(
        "Não encontrei deseq_results_R.csv.\n"
        "Faça upload pelo painel de Arquivos (📁) à esquerda, ou descomente a linha 3."
    )

res_R = pd.read_csv(CAMINHO_R).set_index("gene_id")
print(f"Resultado do R  : {len(res_R):,} genes")
print(f"Resultado do Py : {len(res):,} genes")

# --- CHECAGEM ZERO: as duas trilhas viram as mesmas amostras? -------------
# Se não viram, nada do que vem depois mede implementação — mede sorteio.
print("\n" + "-" * 58)
if os.path.exists(CAMINHO_AMOSTRAS_R):
    ids_R = set(pd.read_csv(CAMINHO_AMOSTRAS_R)["file_id"])
    comuns_am = ids_R & set(meta.index)
    frac = len(comuns_am) / max(len(meta), 1)
    print(f"Amostras em comum R × Python: {len(comuns_am)} de {len(meta)} ({frac:.0%})")
    if frac < 0.99:
        print("\n⚠️ AS TRILHAS ANALISARAM AMOSTRAS DIFERENTES.")
        print("   Tudo abaixo vai medir o sorteio, não a implementação.")
        print("   Volte ao painel, confirme USAR_AMOSTRAS_DO_R = True, e rode")
        print("   o notebook desde a célula M1.")
    else:
        print("✅ Mesmas amostras nas duas trilhas — a comparação é válida.")
else:
    print("⚠️ amostras_R.csv não encontrado — não dá para verificar se as duas")
    print("   trilhas usaram as mesmas amostras. Suba o arquivo pelo painel 📁.")
print("-" * 58)

res_R.head(3)


### 9.1 Os dois testaram os mesmos genes?

Primeira checagem, e a mais banal: se o filtro de genes não foi idêntico, nada
do que vem depois é comparável.

In [ ]:
comuns = res.index.intersection(res_R.index)

print(f"Genes só no R      : {len(res_R.index.difference(res.index)):,}")
print(f"Genes só no Python : {len(res.index.difference(res_R.index)):,}")
print(f"Genes em comum     : {len(comuns):,}")

if len(comuns) == 0:
    raise ValueError("Nenhum gene em comum — os índices não batem. "
                     "Confira se os dois notebooks usaram a mesma matriz.")

C = pd.DataFrame({
    "gene_name": res.loc[comuns, "gene_name"],
    "lfc_py":    res.loc[comuns, "log2FoldChange"],
    "lfc_R":     res_R.loc[comuns, "log2FoldChange"],
    "padj_py":   res.loc[comuns, "padj"],
    "padj_R":    res_R.loc[comuns, "padj"],
}).dropna(subset=["lfc_py", "lfc_R"])

print(f"\nComparáveis (LFC não nulo nos dois): {len(C):,}")

### 9.2 Os fold changes concordam?

In [ ]:
from scipy import stats

r_pearson  = C["lfc_py"].corr(C["lfc_R"])
r_spearman = C["lfc_py"].corr(C["lfc_R"], method="spearman")
dif = (C["lfc_py"] - C["lfc_R"]).abs()

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

lim = np.nanpercentile(np.abs(C[["lfc_py", "lfc_R"]].values), 99.5)
ax[0].scatter(C["lfc_R"], C["lfc_py"], s=5, alpha=.3, c="#1b6ca8")
ax[0].plot([-lim, lim], [-lim, lim], ls="--", c="#d1495b", lw=1.2)
ax[0].set_xlim(-lim, lim); ax[0].set_ylim(-lim, lim)
ax[0].set_xlabel("log2FC — DESeq2 (R)")
ax[0].set_ylabel("log2FC — PyDESeq2 (Python)")
ax[0].set_title(f"Pearson r = {r_pearson:.4f}   ·   Spearman = {r_spearman:.4f}")
ax[0].spines[["top", "right"]].set_visible(False)

ax[1].hist(dif, bins=60, color="#0b7a75", alpha=.85)
ax[1].set_yscale("log")
ax[1].set_xlabel("|diferença absoluta no log2FC|")
ax[1].set_ylabel("nº de genes (log)")
ax[1].set_title("Onde os dois discordam")
ax[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout(); plt.show()

print(f"Diferença mediana : {dif.median():.5f}")
print(f"Percentil 99      : {dif.quantile(.99):.5f}")
print(f"Diferença máxima  : {dif.max():.5f}")

### 9.3 As listas de genes significativos coincidem?

Esta é a comparação que importa de verdade. Correlação alta de fold change é bonita,
mas o que você reporta num artigo é a **lista**.

In [ ]:
# O MESMO critério dos dois lados — e o mesmo que cada trilha usou nos
# próprios gráficos. Com o limiar dentro do teste é só o padj; refiltrar por
# |LFC| aqui compararia listas montadas com uma regra diferente.
def _sig(padj_col, lfc_col):
    ok = C[padj_col].notna() & (C[padj_col] < PADJ)
    if not LIMIAR_NO_TESTE:
        ok &= C[lfc_col].abs() >= LFC
    return set(C.index[ok])

sig_py = _sig("padj_py", "lfc_py")
sig_R  = _sig("padj_R",  "lfc_R")

nos_dois = sig_py & sig_R
so_py    = sig_py - sig_R
so_R     = sig_R  - sig_py
uniao    = sig_py | sig_R

jaccard = len(nos_dois) / len(uniao) if uniao else float("nan")

print(f"DEGs no R          : {len(sig_R):,}")
print(f"DEGs no Python     : {len(sig_py):,}")
print(f"Nos dois           : {len(nos_dois):,}")
print(f"Só no Python       : {len(so_py):,}")
print(f"Só no R            : {len(so_R):,}")
print(f"\nÍndice de Jaccard  : {jaccard:.3f}   (1,0 = listas idênticas)")

# Diagrama de Venn "pobre", mas honesto
fig, ax = plt.subplots(figsize=(8, 2.2))
total = max(len(uniao), 1)
larguras = [len(so_R)/total, len(nos_dois)/total, len(so_py)/total]
esq = 0
for w, cor, rot in zip(larguras, ["#1b6ca8", "#0b7a75", "#d1495b"],
                       ["só R", "ambos", "só Python"]):
    ax.barh(0, w, left=esq, color=cor, height=.55)
    if w > .04:
        ax.text(esq + w/2, 0, f"{rot}\n{int(round(w*total)):,}", ha="center",
                va="center", color="white", fontsize=11, fontweight="bold")
    esq += w
ax.set_xlim(0, 1); ax.axis("off")
ax.set_title("Sobreposição das listas de DEGs", fontsize=13)
plt.tight_layout(); plt.show()

### 9.4 Onde estão as discordâncias?

Se houver genes que aparecem em só uma das listas, olhe **o que eles têm em comum.**

O manual diz: baixa expressão, ou padj perto do limiar. Confira se é isso mesmo —
neste conjunto **não é**, e descobrir por quê é o melhor momento do workshop.

A célula seguinte não olha só os discordantes: ela compara o padj das duas
implementações **em todos os genes**. Se a diferença fosse ruído de convergência,
a razão `padj_py / padj_R` estaria espalhada em torno de 1.


In [ ]:
disc = C.loc[list(so_py | so_R)].copy()

if len(disc) == 0:
    print("As duas listas são idênticas. Não há o que investigar.")
else:
    disc["baseMean"] = res.loc[disc.index, "baseMean"]
    disc["padj_min"] = disc[["padj_py", "padj_R"]].min(axis=1)
    conc = C.loc[list(nos_dois)] if nos_dois else None

    print(f"Genes discordantes: {len(disc):,}\n")
    print(f"baseMean mediana — discordantes : {disc['baseMean'].median():,.1f}")
    if conc is not None and len(conc):
        print(f"baseMean mediana — concordantes : "
              f"{res.loc[conc.index, 'baseMean'].median():,.1f}")
    print(f"\npadj mediano dos discordantes   : {disc['padj_min'].median():.4f}")
    print(f"Quantos com padj entre {PADJ/2:.3f} e {PADJ*2:.3f}? "
          f"{((disc['padj_min'] > PADJ/2) & (disc['padj_min'] < PADJ*2)).sum():,}"
          f"  ({100*((disc['padj_min'] > PADJ/2) & (disc['padj_min'] < PADJ*2)).mean():.0f}%)")

    print("\n--- 10 discordâncias mais expressas ---")
    print(disc.sort_values("baseMean", ascending=False)
              .head(10)[["gene_name", "baseMean", "lfc_R", "lfc_py", "padj_R", "padj_py"]]
              .round(4).to_string())

In [ ]:
# ============================================================================
# 9.4b — A diferença é aleatória ou sistemática?
# ============================================================================
# Ruído de convergência produz razões ESPALHADAS. Uma diferença de método
# produz razões CONCENTRADAS — em torno de algum valor, não necessariamente 1.
# A pergunta certa não é "a razão é 2?", é "a razão é sempre a mesma?".

R = C.dropna(subset=["padj_py", "padj_R"]).copy()
R = R[(R["padj_R"] > 0) & (R["padj_py"] < 1) & (R["padj_R"] < 1)]   # tira o teto em 1
R["razao"] = R["padj_py"] / R["padj_R"]

# ⚠️ Poucos genes entram nesta conta, e isso é esperado: com o limiar dentro
#    do teste, a maioria fica com padj = 1 exato nos dois lados, e 1/1 não diz
#    nada. Só dá para comparar onde os dois deram um p informativo.
n_teto = int((C["padj_py"] >= 1).sum())
print(f"genes com padj informativo nos dois : {len(R):,} de {len(C):,}")
print(f"genes com padj = 1 (fora da conta)  : {n_teto:,}\n")
print(R["razao"].describe(percentiles=[.01, .25, .5, .75, .99]).round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
med = R["razao"].median()
ax[0].hist(R["razao"], bins=80, color="#3d5a80")
ax[0].axvline(1, ls="--", c="0.5", lw=1.2, label="igualdade")
ax[0].axvline(med, ls="--", c="#d1495b", lw=1.6, label=f"mediana = {med:.3f}")
ax[0].set_xlabel("padj (Python) / padj (R)"); ax[0].set_ylabel("nº de genes")
ax[0].set_title("A razão é constante?"); ax[0].legend(frameon=False)

ax[1].scatter(R["padj_R"], R["padj_py"], s=5, alpha=.3, c="#1b6ca8")
lo = max(R["padj_R"].min(), 1e-12)
ax[1].plot([lo, 1], [lo, 1], ls="--", c="0.5", lw=1, label="y = x")
ax[1].plot([lo, 1/med], [med*lo, 1], ls="--", c="#d1495b", lw=1.4,
           label=f"y = {med:.2f}x")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("padj — DESeq2 (R)"); ax[1].set_ylabel("padj — PyDESeq2")
ax[1].set_title("padj × padj, escala log"); ax[1].legend(frameon=False)
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# --- concentrada ou espalhada? --------------------------------------------
# Coeficiente de variação: desvio padrão dividido pela média. Independe da
# escala, então não precisa chutar em torno de que valor a razão vai cair.
cv = R["razao"].std() / R["razao"].mean()
print(f"\nrazão mediana        : {med:.4f}")
print(f"coeficiente de variação : {100*cv:.1f}%   (< 10% = constante)")

if len(R) < 20:
    print(f"""
   ⚠️ Só {len(R)} gene(s) com padj informativo nos dois lados — poucos demais
      para concluir qualquer coisa. Não use este bloco como evidência.""")

elif cv < 0.10:
    # --- de onde vem o valor exato da razão -------------------------------
    # O padj do BH é p × n / rank. Se as duas trilhas testaram números
    # DIFERENTES de genes (filtragem independente distinta), a razão dos padj
    # carrega esse fator junto com a diferença no p bruto.
    n_R  = int(res_R["padj"].notna().sum())
    n_py = int(res["padj"].notna().sum())
    fator_n = n_py / n_R if n_R else float("nan")

    print(f"""
🔍 ACHADO. A razão não está espalhada: variação de {100*cv:.1f}% em torno de
   {med:.3f}, nos {len(R)} genes em que a comparação é possível. Ruído de
   convergência não produz isso.

   A diferença tem DUAS camadas:""")

    print(f"""
   1. O p-valor bruto. As duas bibliotecas calculam de forma diferente o p do
      teste com limiar (lfcThreshold / lfc_null + greaterAbs). Sob a hipótese
      nula composta |LFC| <= limiar há mais de uma forma defensável de somar
      as caudas, e elas escolheram formas diferentes.

   2. O denominador do BH. padj = p × n / rank, onde n é o número de genes
      TESTADOS. A filtragem independente descartou quantidades diferentes:
        R      testou {n_R:,} genes
        Python testou {n_py:,} genes
        n_py / n_R = {fator_n:.4f}

   Se a diferença no p bruto for um fator f, a razão dos padj vira f × {fator_n:.4f}.
   Com f = 2:  2 × {fator_n:.4f} = {2*fator_n:.4f}   (observado: {med:.4f})""")

    if abs(2*fator_n - med) / med < 0.05:
        print("""
   ✅ Bate. As duas camadas explicam o número observado.""")
    else:
        print(f"""
   ⚠️ Não bate exatamente ({2*fator_n:.4f} previsto contra {med:.4f} observado).
      Há mais alguma coisa em jogo além destas duas camadas.""")

    print("""
   Consequências práticas:
     • os log2FC concordam quase perfeitamente — os MODELOS são os mesmos
     • quem decide quantos genes entram na lista é a convenção do p-valor
     • trocar de biblioteca muda a sua tabela sem mudar a sua biologia

   💬 Qual das duas está certa? Não dá para responder olhando o resultado. Dá
      para responder lendo o código-fonte das duas — e é uma pergunta legítima
      para mandar aos mantenedores. Achados assim aparecem quando alguém roda a
      mesma análise duas vezes. É literalmente por isso que este módulo existe.""")

else:
    print(f"""
   A razão varia {100*cv:.1f}% entre os genes — está espalhada, não concentrada.
   Isso é o padrão de diferença por convergência numérica, e não de convenção.
   Siga para a discussão final.""")


In [ ]:
# ============================================================================
# 9.4c — Um caso concreto: o que o encolhimento faz, e por quê
# ============================================================================
# O FGFR3 é o alvo da t(4;14), uma translocação de prognóstico ruim em mieloma.
# Pelo fold change BRUTO ele pareceria um achado. Olhe as contagens e entenda
# por que o apeglm decidiu o contrário — e por que estava certo.

gid = "ENSG00000068078.20"          # FGFR3

if gid in res.index:
    bruto = res_bruto.loc[gid, "log2FoldChange"]
    enc   = res.loc[gid, "log2FoldChange"]
    padj  = res.loc[gid, "padj"]

    print(f"FGFR3 ({gid})\n")
    print(f"  LFC bruto     : {bruto:+.3f}  (SE {res_bruto.loc[gid, 'lfcSE']:.3f})")
    print(f"  LFC encolhido : {enc:+.3f}  (SE {res.loc[gid, 'lfcSE']:.3f})")
    print(f"  {rotulo_padj(padj)}")
    print(f"  encolhimento  : {abs(bruto) - abs(enc):.3f} em log2, "
          f"{100 * (1 - abs(enc) / max(abs(bruto), 1e-9)):.0f}% do efeito")

    print("\n  contagens brutas por grupo:")
    print(pd.DataFrame({"n": counts_f.loc[gid], "g": meta["condition"]})
            .groupby("g")["n"].describe()[["min", "25%", "50%", "75%", "max"]]
            .round(0).to_string())

    print("""
💬 Compare a MEDIANA com o MÁXIMO em cada grupo. A mediana está na casa das
   dezenas; o máximo, na das centenas de milhares. Pouquíssimos pacientes com
   t(4;14) carregam sozinhos toda a diferença.

   O apeglm mede a incerteza dessa estimativa e puxa o efeito para perto de
   zero. Não é o algoritmo escondendo um achado — é ele dizendo que o dado
   não sustenta um efeito de -3,3 em log2.

   Rode plotar_gene("FGFR3") e você vê os pontos isolados no boxplot.""")

    # O caso interessante é quando isso acontece COM padj significativo: a
    # linha da tabela fica com LFC ~ 0 e p pequeno, e parece contraditória.
    # Nem toda rodada tem um. Se houver, mostramos.
    enc_todos = (res_bruto["log2FoldChange"].abs() - res["log2FoldChange"].abs())
    candidatos = res.index[
        eh_significativo(res) & (res["log2FoldChange"].abs() < 0.5)
        & (res_bruto["log2FoldChange"].abs() > 2)
    ]
    if len(candidatos):
        print("\n" + "=" * 62)
        print("E aqui a mesma coisa acontecendo em genes SIGNIFICATIVOS —")
        print("a linha da tabela parece se contradizer: LFC ~ 0 com p pequeno.\n")
        t = pd.DataFrame({
            "gene_name":  res.loc[candidatos, "gene_name"],
            "baseMean":   res.loc[candidatos, "baseMean"].round(0),
            "LFC_bruto":  res_bruto.loc[candidatos, "log2FoldChange"].round(3),
            "LFC_encol":  res.loc[candidatos, "log2FoldChange"].round(3),
            "padj":       [fmt_padj(p) for p in res.loc[candidatos, "padj"]],
        })
        print(t.to_string(index=False))
        print("\n   Não é bug: o p vem do LFC bruto, o LFC exibido é o encolhido.")
    else:
        print("\n   (Nesta rodada nenhum gene significativo teve encolhimento")
        print("    extremo — o FGFR3 acima é a demonstração do fenômeno.)")

else:
    print("FGFR3 não sobreviveu ao filtro do M3 nesta execução.")


### 💬 A discussão que fecha o workshop

Você encontrou: **correlação de fold change praticamente perfeita** e **listas de
DEGs que não coincidem**. Essa combinação é o achado, e ela ensina três coisas.

**1. Os modelos concordam; os p-valores, não.** Pearson acima de 0,999 no log2FC
significa que as duas implementações ajustaram o mesmo modelo e chegaram às mesmas
estimativas de efeito. Se a discordância fosse de convergência numérica, apareceria
primeiro no fold change — e não aparece.

**2. A diferença é constante, não aleatória.** A célula 9.4b mostra que a razão
entre os padj tem desvio padrão na quarta casa decimal. Isso não sai de ruído
numérico: sai de uma diferença de convenção no cálculo do p-valor sob a hipótese
nula composta `|LFC| <= limiar`, onde há mais de uma forma defensável de somar as
caudas. **Qual das duas está certa, o dado não diz** — para responder é preciso ler
o código-fonte das duas bibliotecas. Note também que a razão é calculada sobre
poucas dezenas de genes: com o limiar dentro do teste, a maioria fica com
`padj = 1` exato nos dois lados, e 1 dividido por 1 não informa nada.

**3. A lição que fica.** Um gene com `padj = 0,027` numa implementação e `0,055` na
outra é o mesmo gene, com o mesmo efeito, com a mesma incerteza. O que mudou foi o
lado do corte em que ele caiu. Os genes que você deve levar a sério são os que
sobrevivem às duas análises com folga — e os que ficam na fronteira merecem ser
reportados como fronteira, não como sim ou não.

> Rodar duas vezes não é redundância. É o que transforma "o programa disse que é
> significativo" em "eu sei o quanto essa afirmação depende do programa".

**Para o exercício final:** escreva em três linhas o que você reportaria num artigo
sobre os genes discordantes. Depois compare com o colega ao lado.


In [ ]:
# Exporta a tabela comparativa, para o registro
C["concordancia"] = np.select(
    [C.index.isin(nos_dois), C.index.isin(so_py), C.index.isin(so_R)],
    ["ambos", "so_python", "so_R"], default="nenhum")
C.to_csv("concordancia_R_python.csv")

print("Arquivo salvo: concordancia_R_python.csv")
print("\nPara a seção de métodos, o número que se reporta é este:")
print(f"  concordância entre implementações — Jaccard = {jaccard:.3f}, "
      f"Pearson do log2FC = {r_pearson:.4f}")

---
## Fechamento

Você percorreu o caminho completo: **API pública → matriz de contagens → QC →
modelo estatístico → lista de genes → via biológica → registro reprodutível.**

O que levar daqui, em ordem de importância:

1. **O desenho do contraste decide tudo.** Nenhuma sofisticação estatística conserta
   uma comparação que responde à pergunta errada.
2. **Olhe o dado antes de testar o dado.** PCA e tamanho de biblioteca primeiro, sempre.
3. **Desconfie de listas enormes de DEGs.** Muito significativo geralmente significa
   diferença de tecido ou de composição celular.
4. **Tenha um controle positivo.** O contraste por sexo custa dois minutos e valida
   o pipeline inteiro.
5. **Escreva o que você fez enquanto faz.** Release, versões, semente, limiares —
   e limiares definidos *antes* de ver o resultado.
6. **Um achado que depende da implementação não é um achado.** Os genes que importam
   são os que sobrevivem às duas análises, com folga.

---

*Material didático. Dados de acesso aberto do NCI Genomic Data Commons
(estudo MMRF CoMMpass). Pipeline adaptado de `chol-expression-portfolio`.*